# Scikit-learn Baseline Model

This notebook reproduces a traditional machine learning baseline for the binary diabetes risk prediction task.

The goal is to establish a reliable comparison point for the PyTorch models developed later in this project.

The baseline uses a scikit-learn preprocessing pipeline and a balanced Random Forest classifier, with evaluation focused on recall, ROC-AUC, threshold sensitivity, and class imbalance-aware performance.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

## Load Dataset

The dataset is loaded from the local `data/` directory.

The original target variable, `Diabetes_012`, contains three classes:

- 0: No diabetes
- 1: Prediabetes
- 2: Diabetes

Following the previous project, the target is converted into a binary clinical risk prediction task:

- 0: No diabetes
- 1: At risk, including prediabetes or diabetes

In [2]:
DATA_PATH = Path("../data/diabetes_012_health_indicators_BRFSS2015.csv")

def load_diabetes_data(data_path=DATA_PATH):
    if not DATA_PATH.is_file():
        raise FileNotFoundError(
            f"Dataset not found at {data_path}."
            "Please place the BRFSS 2015 diabetes csv file in the data/ directory."
        )
    return pd.read_csv(data_path)

df = load_diabetes_data()

print(df.shape)
print(df.head())

(253680, 22)
   Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0           0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1           0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2           0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3           0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4           0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0      15.0       1.0  0.0   9.0        

In [3]:
# Convert the original 3-class target into a binary clinical risk label
df["Diabetes_binary"] = (df["Diabetes_012"] > 0).astype(int)

target_distribution = df["Diabetes_binary"].value_counts(normalize=True).sort_index()

target_distribution

Diabetes_binary
0    0.842412
1    0.157588
Name: proportion, dtype: float64

## Train/Test Split

A stratified train/test split is used to preserve the proportion of non-risk and at-risk individuals in both sets.

This is important because the scikit-learn baseline and PyTorch models should be evaluated under the same class distribution. Keeping the test set consistent makes future model comparisons more fair.

Although the binary target and stratified split were introduced in the data understanding notebook, they are recreated here so that this baseline notebook can run independently and provide a reproducible training pipeline.

In [4]:
X = df.drop(columns=["Diabetes_012", "Diabetes_binary"])
y = df["Diabetes_binary"]

# Use stratification to preserve the at-risk class proportion in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (202944, 21)
X_test: (50736, 21)


In [5]:
split_distribution = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index()
})

split_distribution

,train,test
Diabetes_binary,,
0,0.84241,0.84242
1,0.15759,0.15758


## Preprocessing Pipeline

The preprocessing pipeline follows the previous scikit-learn project:

- `BMI` is log-transformed and standardized because it is a skewed numerical feature.
- Ordinal and continuous features are standardized.
- Binary features are passed through without scaling.

The transformations are wrapped in a scikit-learn `ColumnTransformer` to reduce data leakage risk and keep preprocessing consistent between training and evaluation.

In [6]:
skewed_features = ["BMI"]

num_ord_features = [
    "GenHlth", "MentHlth", "PhysHlth",
    "Age", "Education", "Income"
]

binary_features = [
    "HighBP", "HighChol", "CholCheck",
    "Smoker", "Stroke", "HeartDiseaseorAttack",
    "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare",
    "NoDocbcCost", "DiffWalk", "Sex"
]

all_features = skewed_features + num_ord_features + binary_features

print("Number of selected features:", len(all_features))

Number of selected features: 21


In [7]:
# Apply log transformation only to BMI, which is a skewed numerical feature
preprocessor = ColumnTransformer(
    transformers=[
        (
            "skewed", Pipeline([
                ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                ("scaler", StandardScaler())
            ]),
            skewed_features
        ),
        (
            "num_ord",
            StandardScaler(),
            num_ord_features
        ),
        (
            "binary",
            "passthrough",
            binary_features
        )
    ]
)

## Random Forest Baseline

A Random Forest model is used as the main traditional machine learning baseline.

The model is wrapped in a scikit-learn `Pipeline` so that preprocessing and model training are performed together. This helps prevent data leakage and ensures that the same transformations are applied consistently during training and evaluation.

Class weighting is used to reduce the effect of class imbalance by assigning higher penalty to mistakes on the minority class.

In [33]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced", # Penalize minority-class mistakes more heavily
        random_state=42,
        n_jobs=-1
    ))
])

## Hyperparameter Tuning with RandomizedSearchCV

To create a stronger traditional machine learning baseline, `RandomizedSearchCV` is used to tune the Random Forest hyperparameters.

The search space follows the previous scikit-learn deployment project to keep the baseline consistent with the earlier work.

ROC-AUC is used as the optimization metric because the task is imbalanced and probability ranking is more informative than raw accuracy for later threshold tuning.

In [34]:
param_dist = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [10, 15, 20, None],
    "classifier__min_samples_split": [2, 5, 10]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=5,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_rf_pipeline = random_search.best_estimator_

print("Best CV ROC-AUC:", random_search.best_score_)
print("Best parameters:")
print(random_search.best_params_)

Best CV ROC-AUC: 0.8190753243236997
Best parameters:
{'classifier__n_estimators': 200, 'classifier__min_samples_split': 10, 'classifier__max_depth': 15}


## Cross-Validated Training Evaluation at Default Threshold

The first evaluation uses the default probability threshold of 0.5.

This provides a standard baseline before applying recall-focused threshold tuning.

In [25]:
def evaluate_predictions(y_true, y_pred, y_proba):
    """
    Return core classification metrics for binary risk prediction.
    
    ROC-AUC uses predicted probabilities, while the other metrics use
    thresholded class predictions.
    """
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba)
    }

    return {metric: round(score, 4) for metric, score in metrics.items()}

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Generate out-of-fold probabilities from the training set only.
# These predictions are used for threshold analysis without touching the test set.
y_proba_train_cv = cross_val_predict(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

y_pred_train_cv_default = (y_proba_train_cv >= 0.5).astype(int)

In [26]:
metrics_cv_default = evaluate_predictions(
    y_true=y_train,
    y_pred=y_pred_train_cv_default,
    y_proba=y_proba_train_cv
)

metrics_cv_default

{'Accuracy': 0.8402,
 'Precision': 0.4821,
 'Recall': 0.192,
 'F1': 0.2746,
 'ROC-AUC': 0.7935}

In [28]:
print(classification_report(y_train, y_pred_train_cv_default))

              precision    recall  f1-score   support

           0       0.86      0.96      0.91    170962
           1       0.48      0.19      0.27     31982

    accuracy                           0.84    202944
   macro avg       0.67      0.58      0.59    202944
weighted avg       0.80      0.84      0.81    202944



In [29]:
confusion_matrix(y_train, y_pred_train_cv_default)

array([[164367,   6595],
       [ 25843,   6139]])

At the default threshold of 0.5, the model achieves high overall accuracy, but recall for the at-risk class remains low.

This indicates that many at-risk individuals are still classified as non-risk. In a preventative screening context, this is problematic because false negatives are more costly than false positives.

This motivates the threshold tuning step that follows.

## Threshold Tuning with Cross-Validated Training Predictions

The default threshold of 0.5 may not be appropriate for preventative clinical screening.

To avoid tuning the threshold directly on the test set, cross-validated predicted probabilities are generated from the training set. Youden's J statistic is then used to select a threshold that balances sensitivity and specificity.

The selected threshold is later applied once to the held-out test set.

In [31]:
fpr, tpr, thresholds = roc_curve(y_train, y_proba_train_cv)

# Youden's J = sensitivity - false positive rate.
# The selected threshold balances sensitivity and specificity on CV predictions.
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_threshold = thresholds[best_idx]

print(best_threshold)

0.165
